<a href="https://colab.research.google.com/github/emmanuelmassawe/fine-tunning-llm-with-lora-and-qlora/blob/main/fine_tunning_llm_with_lora_and_qlora_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#this is llm fine-tuning using lora and qlora

##install necessary packages

In [31]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate

##import neccesary libraries

In [32]:
import gc
import os

import torch
from datasets import load_dataset
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

In [41]:
import torch
print("GPU ipo:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Jina:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

GPU ipo: True
Jina: Tesla T4
Mounted at /content/drive


In [42]:
## CONFIG: badilisha hapa tu
MODE = "qlora"                       # "lora" (16-bit) au "qlora" (4-bit)
MODEL_NAME = "Qwen/Qwen2.5-0.5B"     # model ndogo kwa majaribio
DATASET = "yahma/alpaca-cleaned"     # dataset ya Hugging Face
DATA_FILE = None                     # mfano: "data.json" ukitumia faili lako (inapita DATASET)
MAX_SAMPLES = 1000
MAX_LENGTH = 512
EPOCHS = 1
LR = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 8
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MERGE_AFTER_TRAINING = False

In [50]:
# T4 haina bf16, kwa hiyo tunachagua kiotomatiki
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float16

In [51]:
# ---------------------------------------------------------------
# HATUA 1: Pakia model (hapa ndipo quantization inapotokea)
# ---------------------------------------------------------------
def load_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if MODE == "qlora":
        # Uzito wa model unahifadhiwa kwa biti 4
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",         # NormalFloat4
            bnb_4bit_use_double_quant=True,    # quantize hata constants
            bnb_4bit_compute_dtype=DTYPE,      # hesabu kwa 16-bit
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=bnb_config, device_map="auto"
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, torch_dtype=DTYPE, device_map="auto"
        )
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()

    model.config.use_cache = False
    return model, tokenizer


In [46]:
# ---------------------------------------------------------------
# HATUA 2: Weka adapter za LoRA
# ---------------------------------------------------------------
def add_lora(model):
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules="all-linear",
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

    # Parameters za LoRA ziwe fp32 ili fp16 training isipate hitilafu
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    model.print_trainable_parameters()
    return model

In [47]:
# ---------------------------------------------------------------
# HATUA 3: Andaa dataset (safu: instruction, input (si lazima), output)
# ---------------------------------------------------------------
def build_dataset(tokenizer):
    if DATA_FILE:
        ds = load_dataset("json", data_files=DATA_FILE, split="train")
    else:
        ds = load_dataset(DATASET, split="train")

    ds = ds.shuffle(seed=42).select(range(min(MAX_SAMPLES, len(ds))))

    def format_example(ex):
        if ex.get("input"):
            prompt = (
                f"### Instruction:\n{ex['instruction']}\n\n"
                f"### Input:\n{ex['input']}\n\n### Response:\n"
            )
        else:
            prompt = f"### Instruction:\n{ex['instruction']}\n\n### Response:\n"
        text = prompt + ex["output"] + tokenizer.eos_token
        return tokenizer(text, truncation=True, max_length=MAX_LENGTH)

    ds = ds.map(format_example, remove_columns=ds.column_names)
    print("Idadi ya mifano:", len(ds))
    return ds

In [48]:
# ---------------------------------------------------------------
# HATUA 4: Train
# ---------------------------------------------------------------
def train(model, tokenizer, dataset):
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=10,
        save_strategy="epoch",
        bf16=(DTYPE == torch.bfloat16),
        fp16=(DTYPE == torch.float16),
        optim="paged_adamw_8bit" if MODE == "qlora" else "adamw_torch",
        gradient_checkpointing=True,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    trainer.train()

In [52]:
 # Hifadhi adapter tu (MB chache), si model nzima
model.save_pretrained(output_dir)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapter imehifadhiwa kwenye:", OUTPUT_DIR)

NameError: name 'model' is not defined